In [1]:
import h5py, numpy as np

DATA = "data"

with h5py.File(f"{DATA}/vg150/VG-SGG.h5", "r") as f:
    boxes = f['boxes_1024'][:]
    active = f['active_object_mask'][:].flatten()
    first_b = f['img_to_first_box'][:]
    last_b = f['img_to_last_box'][:]
    rels = f['relationships'][:]
    first_r = f['img_to_first_rel'][:]
    last_r = f['img_to_last_rel'][:]

w, h = boxes[:,2], boxes[:,3]
areas = w * h
n_boxes = len(boxes)
print(f"{n_boxes} boxes, {len(rels)} rels")


1145398 boxes, 622705 rels


In [2]:
# bad box check
zero_w, zero_h = (w==0).sum(), (h==0).sum()
print(f"zero w: {zero_w}, zero h: {zero_h}")
print(f"negative w: {(w<0).sum()}, negative h: {(h<0).sum()}")
print(f"active_mask true: {active.sum()} / {len(boxes)}")
print(f"tiny boxes (<4px): {(areas < 4).sum()}")


zero w: 0, zero h: 0
negative w: 0, negative h: 0
active_mask true: 1145398 / 1145398
tiny boxes (<4px): 0


In [3]:
# relationship validity
bad_rel = 0
for i in range(min(len(rels), 10000)):
    sid, oid = rels[i]
    if sid >= n_boxes or oid >= n_boxes:
        bad_rel += 1
print(f"bad rels (out of bounds): {bad_rel} / 10000")


bad rels (out of bounds): 0 / 10000


In [4]:
# images with only 1 active object
single = 0
for i in range(len(first_b)):
    s, e = first_b[i], last_b[i] + 1
    if e <= s: continue
    n = active[s:e].sum()
    if n == 1: single += 1
print(f"images with 1 active obj: {single}")


images with 1 active obj: 1910
